# آزمایش جدید تشخیص احساس گفتار

این نوت‌بوک RAVDESS گفتاری و TESS رسمی را دریافت می‌کند، منشأ و فایل‌های تکراری را بررسی می‌کند و آزمایش چهارکلاسه را با گوینده‌های مستقل می‌سازد. نتایج آن بازتولید عدد ۷۴٫۴٪ یا ۸۰٫۳۶٪ مقاله قدیمی نیست.

۲۲۷۲ فایل اصلی چهارکلاسه، پس از حذف یک تکرار دقیق، به ۲۲۷۱ فایل یکتا می‌رسند: ۱۲۴۷ آموزش، ۱۱۲ validation و ۹۱۲ آزمون. TESS در validation این تقسیم حضور ندارد. امتیاز هر مجموعه و UAR را جداگانه گزارش کنید.

فایل‌های خام تابع مجوز [RAVDESS](https://zenodo.org/records/1188976) و [TESS](https://doi.org/10.5683/SP2/E8H2MF) هستند. داده خام در این نوت‌بوک جاسازی نشده است.

In [ ]:
import sys, subprocess
from pathlib import Path
WORK = Path('/content/ser_v10') if Path('/content').exists() else Path.cwd()/'ser_v10'
WORK.mkdir(parents=True, exist_ok=True)
print('Working directory:', WORK)


## محیط اجرا
از runtime تازه Python 3.12 استفاده کنید. آزمایش همراه روی CPU انجام شده است. نصب بسته‌ها ممکن است چند دقیقه طول بکشد. اگر قبلاً کتابخانه‌های یادگیری ماشین را در همین runtime بارگذاری کرده‌اید، پس از نصب runtime را restart و ادامه دهید.

In [ ]:
requirements = 'tensorflow-cpu==2.20.0\nkeras==3.15.1\nnumpy==2.3.5\nscipy==1.17.0\nscikit-learn==1.8.0\nlibrosa==0.11.0\nsoundfile==0.14.0\njoblib==1.5.3\nmatplotlib==3.10.8\nnbformat==5.10.4\n'
(WORK/'requirements.txt').write_text(requirements)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK/'requirements.txt')], check=True)

## کد کامل آزمایش
سلول زیر فقط فایل‌های کد نسخه جدید را در پوشه کاری می‌نویسد. هیچ فایلی از Drive حذف یا جابه‌جا نمی‌شود.

In [ ]:
SOURCES = {'ser_data_v10.py': '"""Auditable four-class RAVDESS/TESS inventory and offline acoustic features.\n\nThis is a NEW experiment, not a reconstruction of the historical paper results.\nFull-utterance temporal interpolation is deliberately NOT a 2-second streaming\nfeature pipeline. No training-fitted normalization or model training occurs here.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport re\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom typing import Iterable\n\nos.environ.setdefault("OMP_NUM_THREADS", "2")\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nos.environ.setdefault("NUMBA_NUM_THREADS", "2")\n\nimport librosa\nimport numpy as np\nimport soundfile as sf\n\nCLASSES = ["neutral", "happy", "sad", "angry"]\nRAV_EMOTIONS = {"01": "neutral", "03": "happy", "04": "sad", "05": "angry"}\nRAV_PATTERN = re.compile(\n    r"^(03)-(01)-(0[1-8])-(0[12])-(0[12])-(0[12])-(\\d{2})\\.wav$", re.I\n)\nTESS_PATTERN = re.compile(\n    r"^(OAF|YAF)_([A-Za-z][A-Za-z0-9_-]*)_"\n    r"(neutral|happy|sad|angry|disgust|fear|fearful|ps|pleasant_surprise)\\.wav$",\n    re.I,\n)\nFEATURE_CONFIG = {\n    "schema": "ser_features_v10.1",\n    "temporal_scope": "full original utterance; offline classification",\n    "sample_rate": 16000,\n    "mono": "arithmetic mean across native channels before resampling",\n    "resample": "librosa.resample res_type=soxr_hq, fix=True, scale=False",\n    "amplitude_normalization": "none",\n    "pre_emphasis": "none",\n    "n_fft": 512,\n    "win_length": 400,\n    "hop_length": 160,\n    "window": "hamming; scipy periodic convention through librosa",\n    "center": True,\n    "pad_mode": "constant",\n    "n_mels": 128,\n    "fmin": 0.0,\n    "fmax": 8000.0,\n    "mel_htk": False,\n    "mel_norm": "slaney",\n    "power": 2.0,\n    "log_power": "10*log10, ref=1.0, amin=1e-10, top_db=80",\n    "time_interpolation": "linear interpolation over full spectrogram to 128 uniformly spaced bins; endpoint-preserving",\n    "logmel_shape": [128, 128, 1],\n    "mfcc": "40 coefficients including c0 from native-length log-Mel; dct_type=2, norm=ortho, lifter=0",\n    "delta": "width=9; orders1,2; mode=nearest; before time interpolation",\n    "pooling": "mean/std per MFCC coefficient and each delta order (population std)",\n    "pitch": "librosa.yin fmin=65,fmax=500,frame_length=1024,hop_length=160,center=True,pad_mode=constant",\n    "pitch_gate": "RMS>=0.1*utterance_max_RMS and RMS>1e-8 and spectral_flatness<0.5; an acoustic heuristic, not ground-truth voicing",\n    "pitch_pooling": "mean,std,p10,p50,p90 over gated frames; zero if no accepted frames",\n    "energy": "20*log10(max(RMS,1e-8)); frame_length=1024,hop_length=160,center=True,pad_mode=constant; mean,std,p10,p50,p90",\n    "additional_features": ["pitch_gate_fraction", "duration_s"],\n    "acoustic_dimension": 252,\n    "dtype": "float32",\n    "learned_normalization": "NONE; fit standardization on training rows downstream",\n}\n\n\nclass InventoryError(ValueError):\n    """The files cannot safely form the declared manifest."""\n\n\ndef _json_hash(value) -> str:\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\nFEATURE_CONFIG_HASH = _json_hash(FEATURE_CONFIG)\n\n\ndef _dump_json(path: Path, value) -> None:\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef _write_csv(path: Path, rows: list[dict], fields: list[str] | None = None) -> None:\n    if fields is None:\n        if not rows:\n            raise ValueError("Explicit columns are required for an empty CSV")\n        fields = list(rows[0])\n    with path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef parse_ravdess(path: str | Path) -> dict | None:\n    """Return selected canonical audio-only speech metadata, else None."""\n    path = Path(path)\n    match = RAV_PATTERN.fullmatch(path.name)\n    if match is None:\n        return None\n    actor = int(match[7])\n    label = RAV_EMOTIONS.get(match[3])\n    if not 1 <= actor <= 24 or label is None:\n        return None\n    # RAVDESS has no strong-neutral recordings; do not accept an impossible ID.\n    if label == "neutral" and match[4] != "01":\n        return None\n    sample_id = "RAVDESS:" + path.stem.lower()\n    return {\n        "sample_id": sample_id,\n        "corpus": "RAVDESS",\n        "speaker_id": f"RAVDESS:{actor:02d}",\n        "parent_utterance_id": sample_id,\n        "label": label,\n        "y": CLASSES.index(label),\n        "path": str(path.resolve()),\n        "split": "train" if actor <= 16 else "val" if actor <= 20 else "test",\n        "fold": "A",\n    }\n\n\ndef parse_tess(path: str | Path) -> dict | None:\n    """TESS speaker is the original filename prefix, never the emotion folder."""\n    path = Path(path)\n    match = TESS_PATTERN.fullmatch(path.name)\n    if match is None or match[3].lower() not in CLASSES:\n        return None\n    speaker, word, label = match[1].upper(), match[2].lower(), match[3].lower()\n    sample_id = f"TESS:{speaker}_{word}_{label}"\n    return {\n        "sample_id": sample_id,\n        "corpus": "TESS",\n        "speaker_id": "TESS:" + speaker,\n        "parent_utterance_id": sample_id,\n        "label": label,\n        "y": CLASSES.index(label),\n        "path": str(path.resolve()),\n        "split": "train" if speaker == "YAF" else "test",\n        "fold": "A",\n    }\n\n\ndef _audio_identity(path: Path) -> dict:\n    """Hash native decoded samples, rate and shape, so container copies dedup."""\n    audio, sr = sf.read(path, dtype="float32", always_2d=True)\n    if not len(audio) or not np.isfinite(audio).all() or sr <= 0:\n        raise InventoryError(f"Empty/nonfinite/invalid audio: {path}")\n    pcm = np.ascontiguousarray(audio, dtype="<f4")\n    # Canonicalize signed zero without mutating source files.\n    pcm[pcm == 0] = 0.0\n    header = json.dumps({"sample_rate": int(sr), "shape": list(pcm.shape)}, sort_keys=True).encode()\n    waveform_hash = hashlib.sha256(header + b"\\0" + pcm.tobytes()).hexdigest()\n    file_hash = hashlib.sha256(path.read_bytes()).hexdigest()\n    return {\n        "sha256": file_hash,\n        "waveform_sha256": waveform_hash,\n        "duration_s": len(audio) / float(sr),\n        "native_sample_rate": int(sr),\n        "native_channels": int(audio.shape[1]),\n        "native_frames": int(len(audio)),\n    }\n\n\ndef _root_list(values: Iterable[str | Path] | str | Path | None) -> list[Path]:\n    if values is None:\n        return []\n    if isinstance(values, (str, Path)):\n        values = [values]\n    return sorted({Path(value).resolve() for value in values})\n\n\ndef validate_rows(rows: list[dict], require_complete: bool = True) -> list[dict]:\n    if not rows:\n        raise InventoryError("No eligible four-class originals found")\n    if len({row["sample_id"] for row in rows}) != len(rows):\n        raise InventoryError("Manifest sample IDs are not unique")\n    groups = {split: {r["speaker_id"] for r in rows if r["split"] == split} for split in ("train", "val", "test")}\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\n        if groups[a] & groups[b]:\n            raise InventoryError(f"Speaker overlap between {a} and {b}")\n    # Parent grouping remains an independent invariant if derived rows are added later.\n    for key in ("parent_utterance_id", "waveform_sha256"):\n        seen = {}\n        for row in rows:\n            old_split = seen.setdefault(row[key], row["split"])\n            if old_split != row["split"]:\n                raise InventoryError(f"{key} crosses partitions: {row[\'sample_id\']}")\n    counts = []\n    expected_speakers = [("RAVDESS", f"RAVDESS:{a:02d}") for a in range(1, 25)] + [("TESS", "TESS:OAF"), ("TESS", "TESS:YAF")]\n    for corpus, speaker in expected_speakers:\n        split = ("test" if speaker == "TESS:OAF" else "train") if corpus == "TESS" else ("train" if int(speaker[-2:]) <= 16 else "val" if int(speaker[-2:]) <= 20 else "test")\n        for label in CLASSES:\n            actual = sum(r["speaker_id"] == speaker and r["label"] == label for r in rows)\n            expected = 200 if corpus == "TESS" else 4 if label == "neutral" else 8\n            counts.append({"fold": "A", "split": split, "corpus": corpus, "speaker_id": speaker, "label": label, "n": actual, "expected_canonical_n": expected})\n            if require_complete and actual != expected:\n                raise InventoryError(f"Incomplete/noncanonical inventory: {speaker} {label}: {actual}, expected {expected}")\n    if require_complete:\n        for split in groups:\n            if {r["label"] for r in rows if r["split"] == split} != set(CLASSES):\n                raise InventoryError(f"{split} lacks classes")\n    return counts\n\n\ndef build_inventory(ravdess_roots, tess_roots, out: str | Path, require_complete: bool = True) -> list[dict]:\n    """Write an immutable fold-A manifest; explicit identical-copy dedup only.\n\n    Existing nonempty output folders are rejected to preserve frozen inventories.\n    Identical decoded audio with a different label, speaker or split is rejected.\n    Canonical counts are validated BEFORE same-speaker/same-label/same-split\n    waveform aliases are collapsed to their lexicographically first identity.\n    Both the complete canonical inventory and explicit alias log are preserved.\n    """\n    out = Path(out)\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError(f"Output directory must be new or empty: {out}")\n    out.mkdir(parents=True, exist_ok=True)\n    roots = {"RAVDESS": _root_list(ravdess_roots), "TESS": _root_list(tess_roots)}\n    excluded, duplicates, conflicts, rows, aliases = [], [], [], [], []\n    by_id, by_wave, visited = {}, {}, set()\n    try:\n        for corpus, paths in roots.items():\n            parser = parse_ravdess if corpus == "RAVDESS" else parse_tess\n            for root in paths:\n                if not root.is_dir():\n                    raise FileNotFoundError(f"Data root is not a directory: {root}")\n                for path in sorted(root.rglob("*")):\n                    if not path.is_file() or path.suffix.lower() != ".wav":\n                        continue\n                    resolved = path.resolve()\n                    if (corpus, resolved) in visited:\n                        duplicates.append({"path": str(resolved), "kept_path": str(resolved), "sample_id": "", "reason": "same physical path encountered through overlapping roots"})\n                        continue\n                    visited.add((corpus, resolved))\n                    row = parser(path)\n                    if row is None:\n                        excluded.append({"path": str(resolved), "corpus_root": corpus, "reason": "filename/modality/label not eligible canonical four-class original"})\n                        continue\n                    row.update(_audio_identity(path))\n                    prior = by_id.get(row["sample_id"])\n                    if prior:\n                        if prior["waveform_sha256"] != row["waveform_sha256"]:\n                            conflicts.append({"sample_id": row["sample_id"], "path": row["path"], "other_path": prior["path"], "reason": "same identity, different decoded audio"})\n                            raise InventoryError(f"Conflicting audio for one identity: {row[\'sample_id\']}")\n                        duplicates.append({"path": row["path"], "kept_path": prior["path"], "sample_id": row["sample_id"], "reason": "same canonical identity and identical decoded waveform"})\n                        continue\n                    prior_wave = by_wave.get(row["waveform_sha256"])\n                    if prior_wave:\n                        reason = None\n                        if prior_wave["label"] != row["label"]:\n                            reason = "identical decoded audio has conflicting labels"\n                        elif prior_wave["speaker_id"] != row["speaker_id"]:\n                            reason = "identical decoded audio has conflicting speaker identities"\n                        elif prior_wave["split"] != row["split"]:\n                            reason = "identical decoded audio crosses partitions"\n                        if reason:\n                            conflicts.append({"sample_id": row["sample_id"], "path": row["path"], "other_path": prior_wave["path"], "reason": reason})\n                            raise InventoryError(f"{reason}: {row[\'sample_id\']} / {prior_wave[\'sample_id\']}")\n                    by_id[row["sample_id"]] = row\n                    by_wave[row["waveform_sha256"]] = row\n                    rows.append(row)\n        rows.sort(key=lambda row: row["sample_id"])\n        _write_csv(out / "excluded.csv", excluded, ["path", "corpus_root", "reason"])\n        _write_csv(out / "duplicates.csv", duplicates, ["path", "kept_path", "sample_id", "reason"])\n        _write_csv(out / "conflicts.csv", conflicts, ["sample_id", "path", "other_path", "reason"])\n        if rows:\n            _write_csv(out / "manifest_candidate.csv", rows)\n        canonical_rows = list(rows)\n        canonical_counts = validate_rows(canonical_rows, require_complete=require_complete)\n        _write_csv(out / "canonical_manifest.csv", canonical_rows)\n        _write_csv(out / "canonical_split_counts.csv", canonical_counts)\n        unique_rows, representatives = [], {}\n        for row in canonical_rows:  # Already sorted by sample ID: deterministic representative.\n            kept = representatives.get(row["waveform_sha256"])\n            if kept is None:\n                representatives[row["waveform_sha256"]] = row\n                unique_rows.append(row)\n            else:\n                aliases.append({"removed_sample_id": row["sample_id"], "kept_sample_id": kept["sample_id"],\n                                "speaker_id": row["speaker_id"], "label": row["label"], "split": row["split"],\n                                "removed_path": row["path"], "kept_path": kept["path"],\n                                "removed_sha256": row["sha256"], "kept_sha256": kept["sha256"],\n                                "waveform_sha256": row["waveform_sha256"],\n                                "reason": "same speaker/label/split and identical waveform; lexicographically first canonical sample retained"})\n        rows = unique_rows\n        counts = validate_rows(rows, require_complete=False)\n        _write_csv(out / "waveform_aliases.csv", aliases, ["removed_sample_id", "kept_sample_id", "speaker_id", "label", "split", "removed_path", "kept_path", "removed_sha256", "kept_sha256", "waveform_sha256", "reason"])\n        _write_csv(out / "manifest.csv", rows)\n        _write_csv(out / "split_counts.csv", counts)\n        aggregate = []\n        for split in ("train", "val", "test"):\n            for corpus in ("RAVDESS", "TESS"):\n                for label in CLASSES:\n                    aggregate.append({"fold": "A", "split": split, "corpus": corpus, "label": label, "n": sum(r["split"] == split and r["corpus"] == corpus and r["label"] == label for r in rows)})\n        _write_csv(out / "partition_counts.csv", aggregate)\n        canonical = [{key: row[key] for key in ("sample_id", "parent_utterance_id", "corpus", "speaker_id", "label", "split", "fold", "waveform_sha256")} for row in rows]\n        metadata = {\n            "schema": "ser_inventory_v10.2", "status": "new_experiment_not_historical_reproduction",\n            "classes": CLASSES, "n": len(rows), "require_complete": require_complete,\n            "n_duplicates_deduplicated": len(duplicates), "n_excluded": len(excluded),\n            "n_canonical_before_waveform_dedup": len(canonical_rows), "n_waveform_aliases_removed": len(aliases),\n            "canonical_count_gate": "applied to canonical_manifest.csv before same-speaker/same-label/same-split waveform alias deduplication",\n            "waveform_alias_rule": "retain lexicographically first sample ID; reject different speaker, label or partition; preserve waveform_aliases.csv",\n            "roots": {key: [str(path) for path in value] for key, value in roots.items()},\n            "inventory_content_sha256": _json_hash(canonical),\n            "manifest_file_sha256": hashlib.sha256((out / "manifest.csv").read_bytes()).hexdigest(),\n            "feature_configuration": FEATURE_CONFIG, "feature_configuration_sha256": FEATURE_CONFIG_HASH,\n            "fold": {"id": "A", "RAVDESS_train": list(range(1, 17)), "RAVDESS_val": list(range(17, 21)), "RAVDESS_test": list(range(21, 25)), "TESS_train": ["YAF"], "TESS_val": [], "TESS_test": ["OAF"]},\n            "versions": {"python": platform.python_version(), "numpy": np.__version__, "librosa": librosa.__version__, "soundfile": sf.__version__},\n            "limitations": ["TESS has no validation speaker; all validation selection uses held-out RAVDESS speakers", "Full-utterance feature extraction is an offline benchmark, not the live-window system", "Canonical complete four-class inventory must be observed before claiming 2272 usable originals"],\n        }\n        _dump_json(out / "metadata.json", metadata)\n        (out / "manifest_candidate.csv").unlink(missing_ok=True)\n        return rows\n    except Exception as exc:\n        _write_csv(out / "excluded.csv", excluded, ["path", "corpus_root", "reason"])\n        _write_csv(out / "duplicates.csv", duplicates, ["path", "kept_path", "sample_id", "reason"])\n        _write_csv(out / "conflicts.csv", conflicts, ["sample_id", "path", "other_path", "reason"])\n        _dump_json(out / "INVENTORY_FAILED.json", {"error": type(exc).__name__, "message": str(exc), "candidate_n": len(rows)})\n        raise\n\n\ndef load_audio(path: str | Path) -> np.ndarray:\n    audio, sr = sf.read(path, dtype="float32", always_2d=True)\n    return resample_audio(audio, sr)\n\n\ndef resample_audio(audio_native: np.ndarray, sr: int) -> np.ndarray:\n    """Downmix decoded (samples,channels) audio and convert to 16 kHz."""\n    audio = np.asarray(audio_native, dtype=np.float32)\n    if audio.ndim != 2 or not len(audio) or not audio.shape[1] or not np.isfinite(audio).all() or sr <= 0:\n        raise ValueError("Expected nonempty finite (samples,channels) audio and a positive sample rate")\n    mono = audio.mean(axis=1, dtype=np.float32)\n    if sr != 16000:\n        mono = librosa.resample(mono, orig_sr=sr, target_sr=16000, res_type="soxr_hq", fix=True, scale=False)\n    return np.ascontiguousarray(mono, dtype=np.float32)\n\n\ndef _summary5(values: np.ndarray) -> np.ndarray:\n    if not len(values):\n        return np.zeros(5, dtype=np.float32)\n    return np.array([values.mean(), values.std(), *np.percentile(values, [10, 50, 90])], dtype=np.float32)\n\n\ndef acoustic_feature_names() -> list[str]:\n    names = []\n    for feature in ("mfcc", "delta1", "delta2"):\n        for stat in ("mean", "std"):\n            names.extend(f"{feature}_{coefficient:02d}_{stat}" for coefficient in range(40))\n    names.extend(f"pitch_hz_{stat}" for stat in ("mean", "std", "p10", "p50", "p90"))\n    names.extend(f"rms_db_{stat}" for stat in ("mean", "std", "p10", "p50", "p90"))\n    names.extend(["pitch_gate_fraction", "duration_s"])\n    return names\n\n\ndef features_for_file(path: str | Path) -> tuple[np.ndarray, np.ndarray]:\n    """Return (128,128,1) log-Mel and (252,) acoustic statistics."""\n    return features_for_audio(load_audio(path))\n\n\ndef features_for_audio(audio_16k: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Same extractor for a decoded/downmixed/resampled 16-kHz mono waveform."""\n    audio = np.asarray(audio_16k, dtype=np.float32)\n    if audio.ndim != 1 or not len(audio) or not np.isfinite(audio).all():\n        raise ValueError("Expected a nonempty finite 16-kHz mono waveform")\n    power = librosa.feature.melspectrogram(\n        y=audio, sr=16000, n_fft=512, win_length=400, hop_length=160,\n        window="hamming", center=True, pad_mode="constant", power=2.0,\n        n_mels=128, fmin=0.0, fmax=8000.0, htk=False, norm="slaney", dtype=np.float32,\n    )\n    logmel_native = librosa.power_to_db(power, ref=1.0, amin=1e-10, top_db=80.0)\n    source_axis = np.linspace(0.0, 1.0, logmel_native.shape[1])\n    target_axis = np.linspace(0.0, 1.0, 128)\n    logmel = np.stack([np.interp(target_axis, source_axis, band) for band in logmel_native]).astype(np.float32)[..., None]\n    mfcc = librosa.feature.mfcc(S=logmel_native, n_mfcc=40, dct_type=2, norm="ortho", lifter=0)\n    parts = [mfcc, librosa.feature.delta(mfcc, width=9, order=1, mode="nearest"), librosa.feature.delta(mfcc, width=9, order=2, mode="nearest")]\n    vector = [stat for part in parts for stat in (part.mean(axis=1), part.std(axis=1))]\n    rms = librosa.feature.rms(y=audio, frame_length=1024, hop_length=160, center=True, pad_mode="constant")[0]\n    energy_db = 20.0 * np.log10(np.maximum(rms, 1e-8))\n    spectrum = np.abs(librosa.stft(audio, n_fft=1024, win_length=1024, hop_length=160, window="hann", center=True, pad_mode="constant"))\n    flatness = librosa.feature.spectral_flatness(S=spectrum, power=2.0)[0]\n    if float(np.max(np.abs(audio))) > 1e-8:\n        pitch = librosa.yin(audio, fmin=65.0, fmax=500.0, sr=16000, frame_length=1024, hop_length=160, center=True, pad_mode="constant")\n        if not (len(pitch) == len(rms) == len(flatness)):\n            raise ValueError("Acoustic frame alignment mismatch")\n        gate = (rms >= 0.1 * rms.max()) & (rms > 1e-8) & (flatness < 0.5) & np.isfinite(pitch)\n        pitch_stats = _summary5(pitch[gate])\n        gate_fraction = float(gate.mean())\n    else:\n        pitch_stats = np.zeros(5, dtype=np.float32)\n        gate_fraction = 0.0\n    vector.extend([pitch_stats, _summary5(energy_db), np.array([gate_fraction, len(audio) / 16000.0], dtype=np.float32)])\n    acoustic = np.concatenate(vector).astype(np.float32)\n    if logmel.shape != (128, 128, 1) or acoustic.shape != (252,) or not (np.isfinite(logmel).all() and np.isfinite(acoustic).all()):\n        raise ValueError(f"Invalid feature output: {logmel.shape}, {acoustic.shape}")\n    return logmel, acoustic\n\n\ndef extract_features(rows: list[dict], out: str | Path, workers: int = 2) -> dict:\n    """Cache all features in exact manifest row order without fitting a scaler."""\n    if not rows:\n        raise ValueError("Cannot extract an empty manifest")\n    if not 1 <= workers <= 4:\n        raise ValueError("Use 1–4 CPU extraction workers")\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    if (out / "features.npz").exists():\n        raise FileExistsError(f"Feature cache already exists: {out / \'features.npz\'}")\n    logmel = np.empty((len(rows), 128, 128, 1), dtype=np.float32)\n    acoustic = np.empty((len(rows), 252), dtype=np.float32)\n    # Warm lazy imports/Numba in one thread before launching modest concurrency.\n    logmel[0], acoustic[0] = features_for_file(rows[0]["path"])\n    print(f"FEATURES 1/{len(rows)}", flush=True)\n    try:\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            results = pool.map(features_for_file, [row["path"] for row in rows[1:]])\n            for index, (mel_row, acoustic_row) in enumerate(results, start=1):\n                logmel[index], acoustic[index] = mel_row, acoustic_row\n                if (index + 1) % 100 == 0 or index + 1 == len(rows):\n                    print(f"FEATURES {index + 1}/{len(rows)}", flush=True)\n        temporary = out / "features_incomplete.npz"\n        np.savez_compressed(temporary, logmel=logmel, acoustic=acoustic,\n                            y=np.array([int(row["y"]) for row in rows], dtype=np.int64),\n                            sample_ids=np.array([row["sample_id"] for row in rows], dtype=str),\n                            feature_config_sha256=np.array(FEATURE_CONFIG_HASH),\n                            class_names=np.array(CLASSES, dtype=str))\n        temporary.replace(out / "features.npz")\n        info = {"n": len(rows), "logmel_shape": list(logmel.shape), "acoustic_shape": list(acoustic.shape),\n                "feature_configuration": FEATURE_CONFIG, "feature_configuration_sha256": FEATURE_CONFIG_HASH,\n                "acoustic_feature_names": acoustic_feature_names(), "workers": workers,\n                "features_sha256": hashlib.sha256((out / "features.npz").read_bytes()).hexdigest(),\n                "row_order": "exactly manifest.csv order; verify sample_ids before training",\n                "normalization": "none fitted here; downstream scaler must fit training rows only"}\n        _dump_json(out / "feature_metadata.json", info)\n        return info\n    except Exception as exc:\n        _dump_json(out / "FEATURES_FAILED.json", {"error": type(exc).__name__, "message": str(exc)})\n        raise\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--ravdess", nargs="+", required=True, help="One or more roots containing canonical RAVDESS WAVs")\n    parser.add_argument("--tess", nargs="+", required=True, help="One or more roots containing canonical TESS WAVs")\n    parser.add_argument("--out", required=True, help="New/empty output directory")\n    parser.add_argument("--allow-incomplete", action="store_true", help="For explicitly labeled partial/synthetic checks; disables canonical count gate")\n    parser.add_argument("--inventory-only", action="store_true")\n    parser.add_argument("--workers", type=int, default=2)\n    args = parser.parse_args()\n    rows = build_inventory(args.ravdess, args.tess, args.out, require_complete=not args.allow_incomplete)\n    print(json.dumps({"inventory_n": len(rows), "classes": CLASSES, "fold": "A", "complete_inventory_required": not args.allow_incomplete}), flush=True)\n    if not args.inventory_only:\n        info = extract_features(rows, args.out, workers=args.workers)\n        print(json.dumps({"features": str(Path(args.out) / "features.npz"), "logmel_shape": info["logmel_shape"], "acoustic_shape": info["acoustic_shape"]}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'ser_models_v10.py': '#!/usr/bin/env python3\n"""Auditable CPU SER experiments. Training never evaluates the test partition.\n\nCommands: train -> freeze -> evaluate. Final evaluation requires an explicit frozen\nselection file. Acoustic and log-Mel features cover the same full utterances.\n"""\nfrom __future__ import annotations\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\nos.environ.setdefault("OMP_NUM_THREADS", "2")\nos.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")\nos.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")\n\nimport joblib\nimport numpy as np\nimport sklearn\nfrom sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\n\nLABELS = ["neutral", "happy", "sad", "angry"]\nCONFIGS = {\n    "cnn": {"residual": False, "clip": False, "scheduler": False},\n    "r0": {"residual": True, "clip": False, "scheduler": False},\n    "r1": {"residual": True, "clip": True, "scheduler": False},\n    "r2": {"residual": True, "clip": False, "scheduler": True},\n    "r3": {"residual": True, "clip": True, "scheduler": True},\n}\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False) + "\\n")\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for chunk in iter(lambda: fh.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_csv(path, rows):\n    if not rows:\n        return\n    with open(path, "w", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef load_data(features_path, manifest_path):\n    with open(manifest_path, newline="") as fh:\n        rows = list(csv.DictReader(fh))\n    with np.load(features_path, allow_pickle=False) as archive:\n        data = {key: archive[key] for key in ("logmel", "acoustic", "y", "sample_ids")}\n    n = len(rows)\n    if any(len(value) != n for value in data.values()):\n        raise ValueError("Feature arrays and manifest have different row counts")\n    ids = [row["sample_id"] for row in rows]\n    if len(set(ids)) != n or ids != data["sample_ids"].tolist():\n        raise ValueError("Manifest IDs must be unique and match feature IDs in order")\n    if data["logmel"].shape[1:] != (128, 128, 1):\n        raise ValueError("Expected logmel shape N x 128 x 128 x 1")\n    if data["acoustic"].ndim != 2 or not np.array_equal(data["y"], [int(row["y"]) for row in rows]):\n        raise ValueError("Acoustic shape or label alignment invalid")\n    if set(np.unique(data["y"])) != set(range(4)):\n        raise ValueError("Expected all four label IDs 0,1,2,3")\n    groups = {}\n    for row in rows:\n        if row["split"] not in ("train", "val", "test"):\n            raise ValueError("Unknown split")\n        for field in ("speaker_id", "parent_utterance_id"):\n            key = (field, row["corpus"], row[field])\n            groups.setdefault(key, set()).add(row["split"])\n    if any(len(value) != 1 for value in groups.values()):\n        raise ValueError("Speaker or original utterance crosses partitions")\n    indices = {split: np.array([i for i, row in enumerate(rows) if row["split"] == split])\n               for split in ("train", "val", "test")}\n    for split, idx in indices.items():\n        if not len(idx) or set(np.unique(data["y"][idx])) != set(range(4)):\n            raise ValueError(f"{split} must have all four classes")\n    return data, rows, indices\n\n\ndef metrics(y, pred):\n    p, r, f, support = precision_recall_fscore_support(y, pred, labels=range(4), zero_division=0)\n    return {\n        "n": int(len(y)), "accuracy": float(accuracy_score(y, pred)),\n        "uar": float(np.mean(r)), "macro_f1": float(np.mean(f)),\n        "weighted_f1": float(np.average(f, weights=support)),\n        "confusion_matrix": confusion_matrix(y, pred, labels=range(4)).tolist(),\n        "classes": {name: {"precision": float(p[i]), "recall": float(r[i]),\n                            "f1": float(f[i]), "support": int(support[i])}\n                    for i, name in enumerate(LABELS)},\n    }\n\n\ndef grouped_metrics(y, pred, rows):\n    result = {"pooled": metrics(y, pred), "corpus": {}, "speaker": {}}\n    for field, key in (("corpus", "corpus"), ("speaker_id", "speaker")):\n        names = sorted({r[field] for r in rows})\n        for name in names:\n            mask = np.array([row[field] == name for row in rows])\n            result[key][name] = metrics(y[mask], pred[mask])\n    result["corpus_balanced_uar"] = float(np.mean([m["uar"] for m in result["corpus"].values()]))\n    return result\n\n\ndef predictions(path, rows, y, pred, scores, score_kind):\n    output = []\n    for i, row in enumerate(rows):\n        record = {key: row[key] for key in ("sample_id", "corpus", "speaker_id", "split")}\n        record.update(true_label=LABELS[int(y[i])], predicted_label=LABELS[int(pred[i])], score_kind=score_kind)\n        record.update({f"score_{label}": float(scores[i, j]) for j, label in enumerate(LABELS)})\n        output.append(record)\n    write_csv(path, output)\n\n\ndef weights_for_train(y, rows, balance):\n    if balance == "none":\n        return np.ones(len(y), dtype="float32")\n    corpora = sorted({row["corpus"] for row in rows})\n    weights = np.empty(len(y), dtype="float32")\n    for corpus in corpora:\n        for label in range(4):\n            mask = np.array([row["corpus"] == corpus for row in rows]) & (y == label)\n            count = int(mask.sum())\n            if count == 0:\n                raise ValueError("Corpus-class balancing requires every train corpus/class cell")\n            weights[mask] = len(y) / (len(corpora) * 4 * count)\n    return weights\n\n\ndef tensorflow():\n    import tensorflow as tf\n    try:\n        tf.config.threading.set_inter_op_parallelism_threads(2)\n        tf.config.threading.set_intra_op_parallelism_threads(2)\n    except RuntimeError:\n        pass\n    return tf\n\n\ndef build_model(residual, seed=17):\n    tf = tensorflow()\n    tf.keras.utils.set_random_seed(seed)\n    layers = tf.keras.layers\n    inputs = layers.Input((128, 128, 1), name="logmel")\n    x = layers.Conv2D(16, 5, strides=2, padding="same", use_bias=False, name="stem_conv")(inputs)\n    x = layers.BatchNormalization(name="stem_bn")(x)\n    x = layers.Activation("relu", name="stem_relu")(x)\n    x = layers.MaxPool2D(2, name="stem_pool")(x)\n    for stage, width in enumerate((16, 32, 64)):\n        prefix = f"stage{stage + 1}"\n        if stage:\n            # Shared transition in BOTH bodies; skip adds no parameters.\n            x = layers.AveragePooling2D(2, name=f"{prefix}_pool")(x)\n            x = layers.Conv2D(width, 1, use_bias=False, name=f"{prefix}_transition")(x)\n            x = layers.BatchNormalization(name=f"{prefix}_transition_bn")(x)\n            x = layers.Activation("relu", name=f"{prefix}_transition_relu")(x)\n        identity = x\n        x = layers.Conv2D(width, 3, padding="same", use_bias=False, name=f"{prefix}_conv1")(x)\n        x = layers.BatchNormalization(name=f"{prefix}_bn1")(x)\n        x = layers.Activation("relu", name=f"{prefix}_relu1")(x)\n        x = layers.Conv2D(width, 3, padding="same", use_bias=False, name=f"{prefix}_conv2")(x)\n        x = layers.BatchNormalization(name=f"{prefix}_bn2")(x)\n        if residual:\n            x = layers.Add(name=f"{prefix}_identity_add")([identity, x])\n        x = layers.Activation("relu", name=f"{prefix}_relu2")(x)\n    x = layers.GlobalAveragePooling2D(name="global_average")(x)\n    outputs = layers.Dense(4, name="emotion_logits")(x)\n    return tf.keras.Model(inputs, outputs, name="residual_cnn" if residual else "plain_cnn")\n\n\ndef mel_normalize_fit(x):\n    # Per-Mel-band statistics across training utterances and their time bins.\n    mean = np.mean(x, axis=(0, 2), keepdims=True, dtype=np.float64).astype("float32")\n    std = np.std(x, axis=(0, 2), keepdims=True, dtype=np.float64).astype("float32")\n    return mean, np.maximum(std, 1e-6)\n\n\ndef infer_neural(model, x, batch_size=64):\n    return np.concatenate([np.asarray(model(x[i:i + batch_size], training=False))\n                           for i in range(0, len(x), batch_size)])\n\n\ndef softmax(logits):\n    exp = np.exp(logits - logits.max(axis=1, keepdims=True))\n    return exp / exp.sum(axis=1, keepdims=True)\n\n\ndef cross_entropy(y, probabilities):\n    return float(-np.log(np.maximum(probabilities[np.arange(len(y)), y], 1e-12)).mean())\n\n\ndef train_neural(args, data, rows, idx, config, seed, out):\n    tf = tensorflow()\n    tf.keras.backend.clear_session()\n    tf.keras.utils.set_random_seed(seed)\n    try:\n        tf.config.experimental.enable_op_determinism()\n    except Exception:\n        pass\n    rng = np.random.default_rng(seed)\n    train_ids, val_ids = idx["train"], idx["val"]\n    mean, std = mel_normalize_fit(data["logmel"][train_ids])\n    np.savez(out / "normalizer.npz", mean=mean, std=std)\n    xtrain = ((data["logmel"][train_ids] - mean) / std).astype("float32")\n    xval = ((data["logmel"][val_ids] - mean) / std).astype("float32")\n    ytrain = data["y"][train_ids].astype("int64")\n    yval = data["y"][val_ids].astype("int64")\n    if not np.isfinite(xtrain).all() or not np.isfinite(xval).all():\n        raise ValueError("Nonfinite train/validation features")\n    weights = weights_for_train(ytrain, [rows[i] for i in train_ids], args.balance)\n    model = build_model(config["residual"], seed)\n    optargs = {"learning_rate": args.lr}\n    if config["clip"]:\n        optargs["global_clipnorm"] = 1.0\n    optimizer = tf.keras.optimizers.Adam(**optargs)\n\n    @tf.function(reduce_retracing=True)\n    def train_step(x, y, weight):\n        with tf.GradientTape() as tape:\n            logits = model(x, training=True)\n            losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=logits)\n            loss = tf.reduce_mean(losses * weight)\n        gradients = tape.gradient(loss, model.trainable_variables)\n        norm = tf.linalg.global_norm(gradients)\n        tf.debugging.check_numerics(loss, "Nonfinite training loss")\n        tf.debugging.check_numerics(norm, "Nonfinite preclip gradient norm")\n        optimizer.apply_gradients(zip(gradients, model.trainable_variables))\n        correct = tf.reduce_sum(tf.cast(tf.argmax(logits, axis=1, output_type=tf.int64) == y, tf.float32))\n        return loss, norm, correct\n\n    best = (-1.0, -float("inf"))\n    best_epoch = 0\n    stop_wait = 0\n    lr_wait = 0\n    lr_best_uar = -1.0\n    history = []\n    started = time.perf_counter()\n    for epoch in range(1, args.epochs + 1):\n        epoch_start = time.perf_counter()\n        order = rng.permutation(len(xtrain))\n        losses, norms, batches, corrects = [], [], [], []\n        lr_used = float(optimizer.learning_rate.numpy())\n        for begin in range(0, len(order), args.batch_size):\n            batch = order[begin:begin + args.batch_size]\n            loss, norm, correct = train_step(xtrain[batch], ytrain[batch], weights[batch])\n            losses.append(float(loss.numpy()))\n            norms.append(float(norm.numpy()))\n            corrects.append(float(correct.numpy()))\n            batches.append(len(batch))\n        probabilities = softmax(infer_neural(model, xval))\n        val_loss = cross_entropy(yval, probabilities)\n        val_metrics = metrics(yval, probabilities.argmax(axis=1))\n        uar = val_metrics["uar"]\n        criterion = (uar, -val_loss)\n        if criterion > best:\n            best, best_epoch, stop_wait = criterion, epoch, 0\n            model.save(out / "best.keras")\n        else:\n            stop_wait += 1\n        # Plateau uses strict validation UAR improvement; loss tie-break is checkpoint-only.\n        if uar > lr_best_uar:\n            lr_best_uar, lr_wait = uar, 0\n        else:\n            lr_wait += 1\n        if config["scheduler"] and lr_wait >= args.scheduler_patience:\n            optimizer.learning_rate.assign(max(args.min_lr, lr_used * 0.5))\n            lr_wait = 0\n        record = {\n            "epoch": epoch, "train_weighted_loss": float(np.average(losses, weights=batches)),\n            "train_accuracy": float(sum(corrects) / len(xtrain)),\n            "val_loss": val_loss, "val_uar": uar, "val_accuracy": val_metrics["accuracy"],\n            "val_macro_f1": val_metrics["macro_f1"], "lr_used": lr_used,\n            "lr_next": float(optimizer.learning_rate.numpy()),\n            "preclip_gradient_norm_mean": float(np.mean(norms)),\n            "preclip_gradient_norm_p95": float(np.quantile(norms, .95)),\n            "preclip_gradient_norm_max": float(np.max(norms)),\n            "preclip_gradient_norm_fraction_above_1": float(np.mean(np.asarray(norms) > 1)),\n            "epoch_seconds": float(time.perf_counter() - epoch_start),\n        }\n        history.append(record)\n        write_csv(out / "history.csv", history)\n        print(f"{out.name} epoch={epoch:02d} val_UAR={uar:.4f} val_loss={val_loss:.4f} seconds={record[\'epoch_seconds\']:.1f}", flush=True)\n        if stop_wait >= args.patience:\n            break\n    elapsed = time.perf_counter() - started\n    model = tf.keras.models.load_model(out / "best.keras", compile=False)\n    probabilities = softmax(infer_neural(model, xval))\n    val_rows = [rows[i] for i in val_ids]\n    predictions(out / "validation_predictions.csv", val_rows, yval, probabilities.argmax(axis=1), probabilities, "softmax_uncalibrated")\n    result = {\n        "status": "complete", "model_kind": "neural", "seed": seed,\n        "best_epoch": best_epoch, "epochs_completed": len(history),\n        "parameter_count": int(model.count_params()), "training_seconds": elapsed,\n        "validation": grouped_metrics(yval, probabilities.argmax(axis=1), val_rows),\n        "selection_rule": "maximum validation UAR; lower validation CE breaks ties; earliest if equal",\n        "checkpoint_sha256": sha256(out / "best.keras"),\n        "normalizer_sha256": sha256(out / "normalizer.npz"),\n    }\n    write_json(out / "result.json", result)\n    return result\n\n\ndef train_svm(args, data, rows, idx, out):\n    train_ids, val_ids = idx["train"], idx["val"]\n    xtrain, xval = data["acoustic"][train_ids], data["acoustic"][val_ids]\n    ytrain, yval = data["y"][train_ids], data["y"][val_ids]\n    if not np.isfinite(xtrain).all() or not np.isfinite(xval).all():\n        raise ValueError("Nonfinite train/validation acoustic features")\n    weights = weights_for_train(ytrain, [rows[i] for i in train_ids], args.balance)\n    trials = []\n    best = -1.0\n    started = time.perf_counter()\n    for c in (.1, 1.0, 10.0):\n        model = make_pipeline(StandardScaler(), SVC(C=c, kernel="rbf", gamma="scale", probability=False))\n        model.fit(xtrain, ytrain, svc__sample_weight=weights)\n        pred = model.predict(xval)\n        trial = {"C": c, **metrics(yval, pred)}\n        trials.append(trial)\n        if trial["uar"] > best:  # Ascending C means smaller C wins exact ties.\n            best, chosen_c = trial["uar"], c\n            joblib.dump(model, out / "best.joblib")\n    model = joblib.load(out / "best.joblib")\n    pred = model.predict(xval)\n    scores = model.decision_function(xval)\n    val_rows = [rows[i] for i in val_ids]\n    predictions(out / "validation_predictions.csv", val_rows, yval, pred, scores, "svm_ovr_decision_not_probability")\n    result = {\n        "status": "complete", "model_kind": "svm", "seed": None, "chosen_C": chosen_c,\n        "C_grid": [.1, 1., 10.], "gamma": "scale", "training_seconds": time.perf_counter() - started,\n        "validation": grouped_metrics(yval, pred, val_rows),\n        "selection_rule": "maximum validation UAR; smaller C breaks ties",\n        "checkpoint_sha256": sha256(out / "best.joblib"),\n    }\n    write_json(out / "svm_trials.json", trials)\n    write_json(out / "result.json", result)\n    print(f"svm chosen_C={chosen_c} val_UAR={best:.4f}", flush=True)\n    return result\n\n\ndef run_train(args):\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=True)\n    if (out / "FROZEN_SELECTIONS.json").exists():\n        raise RuntimeError("Campaign is frozen. Use a new explicitly exploratory output directory for further development")\n    data, rows, idx = load_data(args.features, args.manifest)\n    setting = {\n        "features_sha256": sha256(args.features), "manifest_sha256": sha256(args.manifest),\n        "experiment_driver_sha256": sha256(__file__),\n        "versions": {"python": platform.python_version(), "numpy": np.__version__,\n                     "scikit_learn": sklearn.__version__, "tensorflow": tensorflow().__version__},\n        "labels": LABELS, "epochs": args.epochs, "patience": args.patience,\n        "batch_size": args.batch_size, "lr": args.lr, "balance": args.balance,\n        "scheduler_patience": args.scheduler_patience, "scheduler_factor": .5, "min_lr": args.min_lr,\n        "input_coverage": "full utterance; 128 temporal bins by deterministic resampling",\n        "augmentation": "none", "normalization": "training-only, per Mel band / acoustic StandardScaler",\n        "selection_data": "val only; no test predictions during train",\n        "architecture": "stem Conv5/16 stride2, BN/ReLU, maxpool2; 16/32/64 stages, two Conv3 each; shared avgpool+Conv1 channel transitions; GAP, Dense4 logits",\n        "residual_contrast": "identical learned layers/parameter counts; identity additions are the sole architectural difference",\n        "early_stop_rule": "patience epochs without checkpoint improvement (UAR then CE tie-break)",\n        "scheduler_rule": "factor 0.5 after patience epochs without strict validation-UAR improvement; zero cooldown",\n        "class_corpus_weights": "N_train/(number_of_corpora*4*training_count_in_corpus_class)" if args.balance == "corpus-class" else "all 1",\n        "hardware": {"platform": platform.platform(), "processor": platform.processor(), "cpu_count": os.cpu_count(), "tensorflow_threads": 2},\n    }\n    existing = out / "campaign_config.json"\n    if existing.exists() and json.loads(existing.read_text()) != setting:\n        raise RuntimeError("Existing campaign settings differ; use a fresh output directory")\n    write_json(existing, setting)\n    planned = [name if name == "svm" else f"{name}_seed{seed}"\n               for name in args.models for seed in ([None] if name == "svm" else args.seeds)]\n    planpath = out / "run_plan.json"\n    if planpath.exists():\n        planned = sorted(set(planned) | set(json.loads(planpath.read_text())["planned_runs"]))\n    write_json(planpath, {"planned_runs": planned})\n    for name in args.models:\n        if name not in CONFIGS and name != "svm":\n            raise ValueError(name)\n        for seed in ([None] if name == "svm" else args.seeds):\n            run = out / (name if seed is None else f"{name}_seed{seed}")\n            run.mkdir(exist_ok=True)\n            if (run / "result.json").exists():\n                print(f"Skipping completed run {run.name}", flush=True)\n                continue\n            if (run / "FAILED.json").exists():\n                raise RuntimeError(f"Run {run.name} failed previously; preserve it and use a fresh campaign for a retry")\n            config = {} if name == "svm" else CONFIGS[name]\n            write_json(run / "config.json", {"name": name, "seed": seed, **config, **setting})\n            try:\n                if name == "svm":\n                    train_svm(args, data, rows, idx, run)\n                else:\n                    train_neural(args, data, rows, idx, config, seed, run)\n            except Exception as exc:\n                write_json(run / "FAILED.json", {"status": "failed", "error_type": type(exc).__name__, "error": str(exc)})\n                print(f"FAILED {run.name}: {type(exc).__name__}: {exc}", flush=True)\n                if args.fail_fast:\n                    raise\n    print("Training complete. No test metrics were computed. Review validation, then run freeze.", flush=True)\n\n\ndef run_freeze(args):\n    root = Path(args.output)\n    target = root / "FROZEN_SELECTIONS.json"\n    if target.exists():\n        raise RuntimeError("Selection already frozen")\n    runs, failed = [], []\n    for configpath in sorted(root.glob("*/config.json")):\n        run = configpath.parent\n        if (run / "result.json").exists():\n            result = json.loads((run / "result.json").read_text())\n            runs.append({"run": run.name, "config_sha256": sha256(configpath),\n                         "checkpoint_sha256": result["checkpoint_sha256"],\n                         "normalizer_sha256": result.get("normalizer_sha256"),\n                         "model_kind": result["model_kind"], "validation_uar": result["validation"]["pooled"]["uar"]})\n        else:\n            failed.append(run.name)\n    if not runs:\n        raise RuntimeError("No completed models to freeze")\n    planned = json.loads((root / "run_plan.json").read_text())["planned_runs"]\n    completed = {run["run"] for run in runs}\n    failed = sorted(set(failed) | (set(planned) - completed))\n    by_config = {}\n    for run in runs:\n        name = run["run"].split("_seed")[0]\n        by_config.setdefault(name, []).append(run["validation_uar"])\n    means = {key: float(np.mean(value)) for key, value in by_config.items()}\n    # All runs are evaluated; this choice designates deployment only, never selects the best seed.\n    frozen = {"campaign_config_sha256": sha256(root / "campaign_config.json"), "runs": runs,\n              "failed_or_incomplete_runs": failed, "mean_validation_uar_by_config": means,\n              "validation_selected_configuration": max(sorted(means), key=lambda name: means[name]),\n              "test_not_used_for_selection": True,\n              "evaluation_rule": "evaluate all frozen successful runs and retain failure list; no best-test-score selection",\n              "frozen_unix_time": time.time()}\n    write_json(target, frozen)\n    print(f"Frozen {len(runs)} successful runs, {len(failed)} failed/incomplete. Configuration chosen by validation: {frozen[\'validation_selected_configuration\']}")\n\n\ndef latency(fn, example, warmups=5, repetitions=30):\n    for _ in range(warmups):\n        fn(example)\n    times = []\n    for _ in range(repetitions):\n        start = time.perf_counter()\n        fn(example)\n        times.append((time.perf_counter() - start) * 1000)\n    return {"scope": "ready-feature batch-one model call; excludes audio acquisition, feature extraction and UI",\n            "warmups": warmups, "repetitions": repetitions, "p50_ms": float(np.median(times)),\n            "p95_ms": float(np.quantile(times, .95)), "raw_ms": times}\n\n\ndef run_evaluate(args):\n    root = Path(args.output)\n    frozen_path = Path(args.frozen or root / "FROZEN_SELECTIONS.json")\n    frozen = json.loads(frozen_path.read_text())\n    config = json.loads((root / "campaign_config.json").read_text())\n    if sha256(root / "campaign_config.json") != frozen["campaign_config_sha256"]:\n        raise RuntimeError("Campaign config changed after freeze")\n    if sha256(args.features) != config["features_sha256"] or sha256(args.manifest) != config["manifest_sha256"]:\n        raise RuntimeError("Evaluation data differs from training manifest/features")\n    data, rows, idx = load_data(args.features, args.manifest)\n    test_ids = idx["test"]\n    y = data["y"][test_ids]\n    test_rows = [rows[i] for i in test_ids]\n    summary = []\n    for entry in frozen["runs"]:\n        run = root / entry["run"]\n        if sha256(run / "config.json") != entry["config_sha256"]:\n            raise RuntimeError(f"Run config changed: {run.name}")\n        model_path = run / ("best.joblib" if entry["model_kind"] == "svm" else "best.keras")\n        if sha256(model_path) != entry["checkpoint_sha256"]:\n            raise RuntimeError(f"Checkpoint changed: {run.name}")\n        if entry["normalizer_sha256"] is not None and sha256(run / "normalizer.npz") != entry["normalizer_sha256"]:\n            raise RuntimeError(f"Training normalizer changed: {run.name}")\n        if entry["model_kind"] == "svm":\n            model = joblib.load(model_path)\n            x = data["acoustic"][test_ids]\n            if not np.isfinite(x).all():\n                raise ValueError("Nonfinite test features")\n            scores, pred = model.decision_function(x), model.predict(x)\n            score_kind = "svm_ovr_decision_not_probability"\n            timing = latency(model.predict, x[:1])\n        else:\n            tf = tensorflow()\n            model = tf.keras.models.load_model(model_path, compile=False)\n            with np.load(run / "normalizer.npz") as archive:\n                mean, std = archive["mean"], archive["std"]\n            x = ((data["logmel"][test_ids] - mean) / std).astype("float32")\n            if not np.isfinite(x).all():\n                raise ValueError("Nonfinite test features")\n            scores = softmax(infer_neural(model, x))\n            pred = scores.argmax(axis=1)\n            score_kind = "softmax_uncalibrated"\n            timing = latency(lambda batch: np.asarray(model(batch, training=False)), x[:1])\n        result = grouped_metrics(y, pred, test_rows)\n        write_json(run / "test_metrics.json", result)\n        write_json(run / "ready_feature_latency.json", timing)\n        predictions(run / "test_predictions.csv", test_rows, y, pred, scores, score_kind)\n        record = {"run": run.name, "configuration": run.name.split("_seed")[0],\n                  "validation_uar": entry["validation_uar"], "test_accuracy": result["pooled"]["accuracy"],\n                  "test_uar": result["pooled"]["uar"], "test_macro_f1": result["pooled"]["macro_f1"],\n                  "test_corpus_balanced_uar": result["corpus_balanced_uar"]}\n        for corpus, value in result["corpus"].items():\n            for measure in ("accuracy", "uar", "macro_f1"):\n                record[f"{corpus}_{measure}"] = value[measure]\n        summary.append(record)\n        print(f"FINAL {run.name}: accuracy={record[\'test_accuracy\']:.4f}, UAR={record[\'test_uar\']:.4f}, corpus-balanced UAR={record[\'test_corpus_balanced_uar\']:.4f}", flush=True)\n    write_csv(root / "test_summary.csv", summary)\n    aggregates = {}\n    for name in sorted({record["configuration"] for record in summary}):\n        records = [record for record in summary if record["configuration"] == name]\n        values = {}\n        for key in records[0]:\n            if isinstance(records[0][key], (float, int)):\n                array = [record[key] for record in records]\n                values[key] = {"mean": float(np.mean(array)), "sample_sd": float(np.std(array, ddof=1)) if len(array) > 1 else None}\n        aggregates[name] = {"successful_run_count": len(records), "scores": values}\n    write_json(root / "test_aggregate.json", {"by_configuration": aggregates,\n               "failed_or_incomplete_runs": frozen["failed_or_incomplete_runs"],\n               "uncertainty_note": "Seed SD measures training variability, not speaker-population uncertainty. One TESS test speaker per fold.",\n               "historical_comparison_note": "New protocol differs from historical 1685-item subset; no direct numerical improvement claim."})\n\n\ndef parser():\n    p = argparse.ArgumentParser(description=__doc__)\n    sub = p.add_subparsers(dest="command", required=True)\n    train = sub.add_parser("train")\n    evaluate = sub.add_parser("evaluate")\n    freeze = sub.add_parser("freeze")\n    for command in (train, evaluate):\n        command.add_argument("--features", required=True)\n        command.add_argument("--manifest", required=True)\n    for command in (train, evaluate, freeze):\n        command.add_argument("--output", required=True)\n    train.add_argument("--models", nargs="+", default=["svm", "cnn", "r0", "r1", "r2", "r3"])\n    train.add_argument("--seeds", nargs="+", type=int, default=[17, 42, 73])\n    train.add_argument("--epochs", type=int, default=25)\n    train.add_argument("--patience", type=int, default=7)\n    train.add_argument("--scheduler-patience", type=int, default=4)\n    train.add_argument("--batch-size", type=int, default=32)\n    train.add_argument("--lr", type=float, default=.001)\n    train.add_argument("--min-lr", type=float, default=1e-7)\n    train.add_argument("--balance", choices=["none", "corpus-class"], default="corpus-class")\n    train.add_argument("--fail-fast", action="store_true")\n    evaluate.add_argument("--frozen")\n    return p\n\n\nif __name__ == "__main__":\n    args = parser().parse_args()\n    {"train": run_train, "freeze": run_freeze, "evaluate": run_evaluate}[args.command](args)\n', 'verify_acquisition.py': '"""Verify official public SER archives and extract only metadata-verified WAV files."""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport struct\nimport zipfile\nimport zlib\nimport soundfile as sf\n\nROOT = Path(__file__).resolve().parent\nDATA = ROOT / "data"\n\n\ndef digest(path: Path, algorithm: str) -> str:\n    result = hashlib.new(algorithm)\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef tess_metadata() -> dict:\n    record = json.loads((ROOT / "tess_record.json").read_text())\n    files = record["data"]["latestVersion"]["files"]\n    result = {item["dataFile"]["filename"]: item["dataFile"] for item in files}\n    assert len(result) == len(files) == 2800\n    # The official download archive uses OAF_bite_neutral.wav, while the record\n    # calls the same file OA_bite_neutral.wav. Its size and MD5 match exactly.\n    # Preserve the downloaded spelling and retain the record name in provenance.\n    if "OA_bite_neutral.wav" in result:\n        assert "OAF_bite_neutral.wav" not in result\n        result["OAF_bite_neutral.wav"] = result.pop("OA_bite_neutral.wav")\n    return result\n\n\ndef audio_headers(paths: list[Path]) -> list[dict]:\n    counts = Counter()\n    for path in paths:\n        info = sf.info(path)\n        counts[(info.samplerate, info.channels, info.subtype, info.format)] += 1\n    return [\n        {"sample_rate_hz": rate, "channels": channels, "sample_subtype": subtype, "container": container, "files": count}\n        for (rate, channels, subtype, container), count in sorted(counts.items())\n    ]\n\n\ndef write_verified_tess(name: str, raw: bytes, metadata: dict) -> None:\n    assert Path(name).name == name and name in metadata, name\n    expected = metadata[name]\n    assert len(raw) == expected["filesize"], name\n    assert hashlib.md5(raw).hexdigest() == expected["checksum"]["value"], name\n    destination = DATA / "TESS" / name\n    destination.parent.mkdir(exist_ok=True)\n    destination.write_bytes(raw)\n\n\ndef recover_partial_tess(archive: Path) -> int:\n    """Read complete ZIP entries from a truncated streaming ZIP; verify every WAV.\n\n    This is only an optional transfer accelerator, not evidence that the archive\n    is complete. A final metadata inventory verification remains mandatory.\n    """\n    metadata = tess_metadata()\n    completed = 0\n    with archive.open("rb") as stream:\n        while True:\n            header = stream.read(30)\n            if len(header) != 30 or header[:4] != b"PK\\x03\\x04":\n                break\n            values = struct.unpack("<4s5H3L2H", header)\n            _, _, flags, method, _, _, crc, compressed_size, _, name_size, extra_size = values\n            name = stream.read(name_size).decode("utf-8")\n            stream.read(extra_size)\n            assert method == 8 and flags & 8, (name, flags, method)\n            inflater = zlib.decompressobj(-15)\n            chunks = []\n            while not inflater.eof:\n                block = stream.read(65536)\n                if not block:\n                    return completed\n                chunks.append(inflater.decompress(block))\n            stream.seek(-len(inflater.unused_data), 1)\n            descriptor = stream.read(16)\n            assert len(descriptor) == 16 and descriptor[:4] == b"PK\\x07\\x08"\n            _, expected_crc, expected_compressed_size, expected_size = struct.unpack("<4s3L", descriptor)\n            raw = b"".join(chunks)\n            assert len(raw) == expected_size and zlib.crc32(raw) == expected_crc, name\n            write_verified_tess(name, raw, metadata)\n            completed += 1\n    return completed\n\n\ndef finish(tess_archive: Path) -> dict:\n    rav_archive = DATA / "RAVDESS_official.zip"\n    rav_md5 = digest(rav_archive, "md5")\n    assert rav_md5 == "bc696df654c87fed845eb13823edef8a"\n    rav_out = (DATA / "RAVDESS").resolve()\n    rav_out.mkdir(exist_ok=True)\n    with zipfile.ZipFile(rav_archive) as archive:\n        for member in archive.infolist():\n            assert (rav_out / member.filename).resolve().is_relative_to(rav_out)\n        archive.extractall(rav_out)\n    rav_files = sorted(rav_out.rglob("*.wav"))\n    assert len(rav_files) == 1440\n    assert len({path.name for path in rav_files}) == 1440\n    assert all(path.name.startswith("03-01-") for path in rav_files)\n    metadata = tess_metadata()\n    seen = set()\n    with zipfile.ZipFile(tess_archive) as archive:\n        for member in archive.infolist():\n            if member.is_dir():\n                continue\n            name = member.filename\n            if name == "MANIFEST.TXT":\n                (ROOT / "TESS_download_manifest.txt").write_bytes(archive.read(member))\n                continue\n            assert name not in seen, name\n            write_verified_tess(name, archive.read(member), metadata)\n            seen.add(name)\n    assert seen == set(metadata), sorted(set(metadata) - seen)\n    tess_files = sorted((DATA / "TESS").glob("*.wav"))\n    assert len(tess_files) == 2800\n    checks = []\n    for path in tess_files:\n        item = metadata[path.name]\n        actual_md5 = digest(path, "md5")\n        assert actual_md5 == item["checksum"]["value"]\n        checks.append({"filename": path.name, "metadata_filename": item["filename"],\n                       "file_id": item["id"], "bytes": path.stat().st_size,\n                       "md5": actual_md5, "official_md5": item["checksum"]["value"]})\n    (ROOT / "tess_verified_files.json").write_text(json.dumps(checks, indent=2) + "\\n")\n    report = {\n        "verified_at_utc": datetime.now(timezone.utc).isoformat(),\n        "source_kind": "official_public_repositories; not private Google Drive derivatives",\n        "RAVDESS": {\n            "record": "https://zenodo.org/records/1188976",\n            "download": "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1",\n            "archive": str(rav_archive), "archive_bytes": rav_archive.stat().st_size,\n            "official_archive_md5": "bc696df654c87fed845eb13823edef8a", "verified_archive_md5": rav_md5,\n            "archive_sha256": digest(rav_archive, "sha256"),\n            "path": str(rav_out), "wav_count": len(rav_files),\n            "audio_header_counts": audio_headers(rav_files),\n            "speakers": dict(sorted(Counter(path.stem.split("-")[-1] for path in rav_files).items())),\n            "modality": "audio-only speech (03-01), all eight original emotion classes",\n        },\n        "TESS": {\n            "record": "https://doi.org/10.5683/SP2/E8H2MF",\n            "metadata": "https://borealisdata.ca/api/datasets/:persistentId/?persistentId=doi:10.5683/SP2/E8H2MF",\n            "download": "https://borealisdata.ca/api/access/dataset/93347",\n            "dataset_version": "1.0", "archive": str(tess_archive),\n            "license": json.loads((ROOT / "tess_record.json").read_text())["data"]["latestVersion"]["license"],\n            "archive_bytes": tess_archive.stat().st_size, "archive_sha256": digest(tess_archive, "sha256"),\n            "path": str(DATA / "TESS"), "wav_count": len(tess_files),\n            "audio_header_counts": audio_headers(tess_files),\n            "all_2800_file_md5_match_official_metadata": True,\n            "filename_aliases": [{"archive_filename": "OAF_bite_neutral.wav",\n                                  "metadata_filename": "OA_bite_neutral.wav", "file_id": 95758,\n                                  "reason": "Official archive spelling differs from metadata; identical MD5 bc32cdf920bb82376687fe228c56bd30 verifies identity."}],\n            "checksums_file": str(ROOT / "tess_verified_files.json"),\n            "speakers": dict(sorted(Counter(path.name.split("_")[0] for path in tess_files).items())),\n            "notes": "All seven original categories acquired; four target classes selected downstream.",\n        },\n    }\n    (ROOT / "acquisition.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--partial", type=Path)\n    parser.add_argument("--tess-archive", type=Path, default=DATA / "TESS_official_complete.zip")\n    args = parser.parse_args()\n    if args.partial:\n        print("Recovered and MD5 verified:", recover_partial_tess(args.partial))\n    else:\n        print(json.dumps(finish(args.tess_archive), indent=2))\n', 'ser_infer.py': '#!/usr/bin/env python3\n"""Full-utterance SER inference and auditable ready-file timing for v10 models.\n\nUses the same feature implementation and saved training normalizers as the\nexperiment. Input must be an already available complete audio file. This is not\na microphone or streaming latency measurement.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport sys\nimport time\n\nfor name, value in {"OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "1",\n                    "NUMBA_NUM_THREADS": "2", "TF_NUM_INTRAOP_THREADS": "2",\n                    "TF_NUM_INTEROP_THREADS": "2", "TF_CPP_MIN_LOG_LEVEL": "2"}.items():\n    os.environ.setdefault(name, value)\n\n# Both a flat released code bundle and the development tree are supported.\nif not (Path(__file__).resolve().parent / "ser_data_v10.py").exists():\n    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "code_review"))\n\nimport joblib\nimport numpy as np\nimport soundfile as sf\nimport ser_data_v10 as features\n\n\ndef sha256(path: str | Path) -> str:\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef read_json(path: str | Path) -> dict:\n    return json.loads(Path(path).read_text())\n\n\ndef softmax(logits: np.ndarray) -> np.ndarray:\n    values = np.exp(logits - logits.max(axis=1, keepdims=True))\n    return values / values.sum(axis=1, keepdims=True)\n\n\nclass Predictor:\n    def __init__(self, model_path: str | Path, feature_metadata: str | Path,\n                 normalizer_path: str | Path | None = None,\n                 frozen_selections: str | Path | None = None,\n                 compiled: bool = True):\n        started = time.perf_counter()\n        self.model_path = Path(model_path).resolve()\n        self.metadata_path = Path(feature_metadata).resolve()\n        metadata = read_json(self.metadata_path)\n        if metadata.get("feature_configuration_sha256") != features.FEATURE_CONFIG_HASH:\n            raise ValueError("Model feature metadata does not match the shared feature implementation")\n        self.artifacts = {"model_path": str(self.model_path),\n                          "model_sha256": sha256(self.model_path),\n                          "feature_metadata_path": str(self.metadata_path),\n                          "feature_metadata_sha256": sha256(self.metadata_path),\n                          "feature_configuration_sha256": features.FEATURE_CONFIG_HASH,\n                          "feature_implementation_sha256": sha256(features.__file__)}\n        config_path = self.model_path.parent / "config.json"\n        if config_path.exists():\n            config = read_json(config_path)\n            if config.get("labels") != features.CLASSES:\n                raise ValueError("Training label order differs from inference label order")\n            if config.get("features_sha256") != metadata.get("features_sha256"):\n                raise ValueError("Feature metadata does not identify the training feature cache")\n            self.artifacts["training_config_sha256"] = sha256(config_path)\n        result_path = self.model_path.parent / "result.json"\n        result = read_json(result_path) if result_path.exists() else {}\n        if result and result.get("checkpoint_sha256") != self.artifacts["model_sha256"]:\n            raise ValueError("Checkpoint hash differs from the completed training result")\n\n        if self.model_path.suffix == ".joblib":\n            self.kind = "svm"\n            from sklearn.pipeline import Pipeline\n            from sklearn.preprocessing import StandardScaler\n            from sklearn.svm import SVC\n            self.model = joblib.load(self.model_path)\n            if not isinstance(self.model, Pipeline) or len(self.model.steps) != 2:\n                raise ValueError("Expected the v10 StandardScaler/SVC pipeline")\n            self.scaler, self.classifier = [step for _, step in self.model.steps]\n            if not isinstance(self.scaler, StandardScaler) or not isinstance(self.classifier, SVC):\n                raise ValueError("Expected StandardScaler followed by SVC")\n            if self.scaler.n_features_in_ != 252 or self.classifier.probability:\n                raise ValueError("Expected 252 acoustic features and uncalibrated SVC")\n            if not np.array_equal(self.classifier.classes_, np.arange(4)):\n                raise ValueError("Expected SVM class IDs 0,1,2,3")\n            self.backend = "scikit-learn saved StandardScaler + SVC"\n        elif self.model_path.suffix == ".keras":\n            self.kind = "neural"\n            import tensorflow as tf\n            self.tf = tf\n            try:\n                tf.config.threading.set_intra_op_parallelism_threads(2)\n                tf.config.threading.set_inter_op_parallelism_threads(2)\n            except RuntimeError:\n                pass\n            normalizer_path = Path(normalizer_path or self.model_path.parent / "normalizer.npz").resolve()\n            normalizer_hash = sha256(normalizer_path)\n            if result and result.get("normalizer_sha256") != normalizer_hash:\n                raise ValueError("Normalizer hash differs from the completed training result")\n            with np.load(normalizer_path, allow_pickle=False) as archive:\n                self.mean, self.std = archive["mean"], archive["std"]\n            if self.mean.shape != (1, 128, 1, 1) or self.std.shape != self.mean.shape:\n                raise ValueError("Expected saved per-Mel-band normalizer shape (1,128,1,1)")\n            if not np.isfinite(self.mean).all() or not np.isfinite(self.std).all() or np.any(self.std <= 0):\n                raise ValueError("Normalizer contains invalid values")\n            self.artifacts.update(normalizer_path=str(normalizer_path), normalizer_sha256=normalizer_hash)\n            self.model = tf.keras.models.load_model(self.model_path, compile=False)\n            if self.model.input_shape != (None, 128, 128, 1) or self.model.output_shape != (None, 4):\n                raise ValueError("Expected v10 log-Mel input and four logits")\n            if compiled:\n                @tf.function(input_signature=[tf.TensorSpec((None, 128, 128, 1), tf.float32)])\n                def infer(x):\n                    return self.model(x, training=False)\n                self.infer = infer\n                self.backend = "TensorFlow tf.function; training=False; output synchronized by .numpy()"\n            else:\n                self.infer = lambda x: self.model(x, training=False)\n                self.backend = "TensorFlow eager; training=False; output synchronized by .numpy()"\n        else:\n            raise ValueError("Model must be a v10 .joblib pipeline or .keras checkpoint")\n\n        frozen_path = Path(frozen_selections) if frozen_selections else self.model_path.parent.parent / "FROZEN_SELECTIONS.json"\n        if frozen_path.exists():\n            frozen = read_json(frozen_path)\n            matches = [entry for entry in frozen["runs"] if entry["run"] == self.model_path.parent.name]\n            if len(matches) != 1:\n                raise ValueError("Model run is absent from the frozen campaign")\n            entry = matches[0]\n            if entry["checkpoint_sha256"] != self.artifacts["model_sha256"]:\n                raise ValueError("Model differs from the frozen checkpoint")\n            if entry.get("normalizer_sha256") != self.artifacts.get("normalizer_sha256"):\n                raise ValueError("Normalizer differs from the frozen checkpoint")\n            if entry["config_sha256"] != self.artifacts.get("training_config_sha256"):\n                raise ValueError("Training configuration differs from the frozen campaign")\n            self.artifacts.update(frozen_selections_path=str(frozen_path.resolve()),\n                                  frozen_selections_sha256=sha256(frozen_path),\n                                  validation_selected_configuration=frozen["validation_selected_configuration"])\n        elif frozen_selections:\n            raise FileNotFoundError(frozen_path)\n        self.initialization_ms = (time.perf_counter() - started) * 1000\n\n    def normalize(self, logmel, acoustic):\n        if self.kind == "svm":\n            return self.scaler.transform(acoustic[None, :])\n        return ((logmel[None, ...] - self.mean) / self.std).astype(np.float32)\n\n    def model_call(self, normalized):\n        if not np.isfinite(normalized).all():\n            raise ValueError("Nonfinite normalized features")\n        if self.kind == "svm":\n            # Calling both preserves SVC\'s vote/tie rule: argmax(OVR scores) is\n            # not a guaranteed substitute for SVC.predict with break_ties=False.\n            return self.classifier.predict(normalized), self.classifier.decision_function(normalized)\n        return np.asarray(self.infer(normalized).numpy())\n\n    def postprocess(self, raw):\n        if self.kind == "svm":\n            pred, scores = raw\n            index = int(pred[0])\n            values = np.asarray(scores)[0]\n            score_kind = "svm_ovr_decision_values_not_probabilities"\n            note = "Uncalibrated SVM decision values; do not interpret as percentages or confidence."\n        else:\n            values = softmax(np.asarray(raw))[0]\n            index = int(values.argmax())\n            score_kind = "uncalibrated_softmax"\n            note = "Softmax scores are uncalibrated; they are not measured accuracy or calibrated confidence."\n        if not np.isfinite(values).all():\n            raise ValueError("Nonfinite model scores")\n        return {"label": features.CLASSES[index], "label_id": index,\n                "score_kind": score_kind,\n                "scores": dict(zip(features.CLASSES, map(float, values))), "score_interpretation": note}\n\n    def predict_file(self, audio_path: str | Path):\n        start = time.perf_counter()\n        native, sr = sf.read(audio_path, dtype="float32", always_2d=True)\n        if len(native) == 0 or not np.isfinite(native).all():\n            raise ValueError("Audio must be nonempty and finite")\n        if np.max(np.abs(native)) <= 1e-8:\n            raise ValueError("The recording contains no usable audio signal")\n        decoded = time.perf_counter()\n        audio = features.resample_audio(native, sr)\n        logmel, acoustic = features.features_for_audio(audio)\n        extracted = time.perf_counter()\n        normalized = self.normalize(logmel, acoustic)\n        standardized = time.perf_counter()\n        raw = self.model_call(normalized)\n        inferred = time.perf_counter()\n        prediction = self.postprocess(raw)\n        done = time.perf_counter()\n        duration = len(native) / sr\n        timing = {"load_decode_ms": (decoded - start) * 1000,\n                  "resample_feature_ms": (extracted - decoded) * 1000,\n                  "normalization_ms": (standardized - extracted) * 1000,\n                  "model_call_ms": (inferred - standardized) * 1000,\n                  "postprocess_ms": (done - inferred) * 1000,\n                  "total_ready_file_ms": (done - start) * 1000,\n                  "processing_rtf": (done - start) / duration}\n        info = {"path": str(Path(audio_path).resolve()), "duration_s": duration,\n                "native_sample_rate_hz": int(sr), "native_channels": int(native.shape[1]),\n                "resampled_duration_s": len(audio) / 16000.0}\n        return prediction, timing, info\n\n\ndef environment(predictor: Predictor) -> dict:\n    import librosa\n    import sklearn\n    import threadpoolctl\n    result = {"platform": platform.platform(), "python": platform.python_version(),\n              "logical_cpu_count": os.cpu_count(), "processor": platform.processor(),\n              "cpu_affinity_count": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,\n              "load_average_1_5_15_minutes": list(os.getloadavg()) if hasattr(os, "getloadavg") else None,\n              "versions": {"numpy": np.__version__, "librosa": librosa.__version__,\n                           "soundfile": sf.__version__, "scikit_learn": sklearn.__version__},\n              "thread_environment": {key: os.environ.get(key) for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMBA_NUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS")},\n              "thread_pools": threadpoolctl.threadpool_info(), "inference_backend": predictor.backend,\n              "cpu_isolation": "No CPU isolation is enforced; concurrent training or other processes can affect timing."}\n    if predictor.kind == "neural":\n        result["versions"]["tensorflow"] = predictor.tf.__version__\n        result["tensorflow_visible_devices"] = [device.device_type for device in predictor.tf.config.get_visible_devices()]\n    return result\n\n\ndef summarize(timings: list[dict]) -> dict:\n    return {name: {"p50": float(np.percentile([row[name] for row in timings], 50)),\n                   "p95": float(np.percentile([row[name] for row in timings], 95)),\n                   "mean": float(np.mean([row[name] for row in timings])),\n                   "minimum": float(min(row[name] for row in timings)),\n                   "maximum": float(max(row[name] for row in timings))}\n            for name in timings[0]}\n\n\ndef run(args) -> dict:\n    predictor = Predictor(args.model, args.feature_metadata, args.normalizer,\n                          args.frozen_selections, compiled=not args.eager)\n    report = {"scope": "Full-utterance offline prediction from an already available audio file",\n              "timing_scope": "Ready-file wall time includes decode, downmix/resample, shared feature extraction, training-fitted normalization, model calls, and score postprocessing. Excludes recording/waiting for speech, process startup, module imports, model initialization, JSON serialization and UI.",\n              "feature_scope": "Shared extractor calculates both log-Mel and acoustic features on every pass, including the unused representation; no cache of features is used.",\n              "rtf_definition": "total_ready_file_seconds / original_audio_duration_seconds; this is processing RTF, not first-prediction or microphone latency",\n              "model_kind": predictor.kind, "labels": features.CLASSES,\n              "artifacts": predictor.artifacts, "initialization_ms": predictor.initialization_ms,\n              "initialization_scope": "Artifact checks, model/normalizer loading and backend initialization; excludes script/module startup. tf.function tracing occurs on the first prediction, if compiled.",\n              "runtime_before": environment(predictor), "workload_note": args.workload_note,\n              "benchmark": bool(args.benchmark)}\n    if args.benchmark:\n        warmups, measured = [], []\n        for _ in range(args.warmups):\n            _, timing, _ = predictor.predict_file(args.audio)\n            warmups.append(timing)\n        for _ in range(args.repeats):\n            prediction, timing, audio_info = predictor.predict_file(args.audio)\n            measured.append(timing)\n        report.update(warmup_count=args.warmups, measured_repetitions=args.repeats,\n                      warmup_raw=warmups, measured_raw=measured, measured_summary=summarize(measured),\n                      summary_scope="Percentiles describe repeated processing of the same file in this process, with OS file caches and warmed libraries; they are not a distribution over speakers or audio files.",\n                      prediction=prediction, audio=audio_info)\n    else:\n        prediction, timing, audio_info = predictor.predict_file(args.audio)\n        report.update(prediction=prediction, timing=timing, audio=audio_info)\n    report["runtime_after"] = environment(predictor)\n    return report\n\n\ndef parser():\n    result = argparse.ArgumentParser(description=__doc__)\n    result.add_argument("audio", help="Complete readable audio file; never treated as a streaming window")\n    result.add_argument("--model", required=True, help="best.joblib or best.keras")\n    result.add_argument("--feature-metadata", required=True, help="feature_metadata.json from the training feature cache")\n    result.add_argument("--normalizer", help="Neural normalizer.npz; defaults beside the checkpoint")\n    result.add_argument("--frozen-selections", help="Optional explicit FROZEN_SELECTIONS.json; auto-detected beside the campaign")\n    result.add_argument("--eager", action="store_true", help="Use eager neural inference instead of tf.function")\n    result.add_argument("--benchmark", action="store_true")\n    result.add_argument("--warmups", type=int, default=3)\n    result.add_argument("--repeats", type=int, default=20)\n    result.add_argument("--workload-note", default="Other concurrent workload was not independently controlled.")\n    result.add_argument("--output", help="JSON destination; stdout when omitted")\n    return result\n\n\ndef main():\n    args = parser().parse_args()\n    if args.warmups < 0 or args.repeats < 1:\n        raise SystemExit("warmups must be nonnegative and repeats must be positive")\n    report = run(args)\n    serialized = json.dumps(report, ensure_ascii=False, allow_nan=False, indent=2) + "\\n"\n    if args.output:\n        destination = Path(args.output)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        destination.write_text(serialized)\n        print(str(destination.resolve()))\n    else:\n        print(serialized, end="")\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, text in SOURCES.items():
    (WORK/name).write_text(text)
print('Source files:', list(SOURCES))

## دریافت داده رسمی
هر دو مجموعه کامل دریافت می‌شوند و سپس فقط چهار کلاس انتخاب می‌شوند. بررسی checksum پیش از استخراج ویژگی اجباری است. دانلود TESS ممکن است چند دقیقه طول بکشد. داده‌ها از منابع عمومی رسمی دریافت می‌شوند و نیازی به اتصال Drive نیست.

In [ ]:
import urllib.request, zipfile, hashlib
DATA=WORK/'data'
DATA.mkdir(exist_ok=True)
def download(url, path):
    if path.exists() and path.stat().st_size:
        print('Existing file; verification follows:', path.name)
        return
    partial=path.with_suffix(path.suffix+'.partial')
    with urllib.request.urlopen(url, timeout=60) as response, partial.open('wb') as f:
        while True:
            chunk=response.read(1024*1024)
            if not chunk: break
            f.write(chunk)
    partial.replace(path)
    print('Downloaded:', path.name, path.stat().st_size)
download('https://borealisdata.ca/api/datasets/:persistentId/?persistentId=doi:10.5683/SP2/E8H2MF', WORK/'tess_record.json')
download('https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1', DATA/'RAVDESS_official.zip')
download('https://borealisdata.ca/api/access/dataset/93347', DATA/'TESS_official_complete.zip')
subprocess.run([sys.executable, str(WORK/'verify_acquisition.py')], check=True)


## تقسیم‌بندی و ویژگی‌ها
شناسه گوینده، برچسب و هش هر فایل ثبت می‌شود. حذف تکرار دقیق مستند است. استانداردسازی آماری در مرحله آموزش فقط با داده آموزش برازش می‌شود. ویژگی Mel تمام طول گفتار را با تغییر اندازه محور زمان در ۱۲۸ فریم نمایش می‌دهد؛ این ورودی یک پنجره زنده ۲۵ میلی‌ثانیه‌ای نیست.

In [ ]:
FEATURES=WORK/'features'
subprocess.run([sys.executable, str(WORK/'ser_data_v10.py'), '--ravdess', str(DATA/'RAVDESS'), '--tess', str(DATA/'TESS'), '--out', str(FEATURES), '--workers', '2'], check=True)
print((FEATURES/'partition_counts.csv').read_text())


## آموزش مقایسه‌ای
CNN و ResNet پارامترها و وزن‌های اولیه یکسان دارند. تفاوت آن‌ها اتصال residual است. r0 بدون کنترل، r1 فقط clipping، r2 فقط scheduler و r3 هر دو را دارد. سه seed برای مدل‌های عصبی اجرا می‌شود. SVM شبکه ازپیش‌آموزش‌دیده نیست و یک خط پایه کلاسیک جداگانه است.

In [ ]:
EXPERIMENT=WORK/'experiments'/'foldA'
subprocess.run([sys.executable, str(WORK/'ser_models_v10.py'), 'train', '--features', str(FEATURES/'features.npz'), '--manifest', str(FEATURES/'manifest.csv'), '--output', str(EXPERIMENT), '--models', 'svm','cnn','r0','r1','r2','r3', '--seeds','17','42','73','--epochs','25'], check=True)


## ثبت انتخاب و آزمون نهایی
پس از پایان تصمیم‌گیری بر اساس validation، مقدار زیر را True کنید. این کار انتخاب‌ها را ثبت و همه اجراهای برنامه‌ریزی‌شده را روی آزمون گزارش می‌کند. پس از مشاهده آزمون، مدل را برای بهتر کردن همان امتیاز تغییر ندهید. هر نتیجه جدید باید همراه با روش انتخاب و محدودیت‌های تقسیم داده گزارش شود.

In [ ]:
RUN_HELD_OUT=False
if RUN_HELD_OUT:
    subprocess.run([sys.executable, str(WORK/'ser_models_v10.py'), 'freeze', '--output', str(EXPERIMENT)], check=True)
    subprocess.run([sys.executable, str(WORK/'ser_models_v10.py'), 'evaluate', '--features', str(FEATURES/'features.npz'), '--manifest', str(FEATURES/'manifest.csv'), '--output', str(EXPERIMENT)], check=True)
else:
    print('Test remains unevaluated. Review validation first.')


## نگهداری خروجی‌ها
مدل‌ها، گزارش‌ها و شناسه‌های تقسیم‌بندی ذخیره می‌شوند. آرشیو زیر شامل فایل‌های خام صوتی نیست. runtime Colab موقت است؛ آرشیو را دانلود و نگهداری کنید. برای baseline جدید emotion2vec و نسخه regularized به کد و دستورالعمل جداگانه بسته پروژه مراجعه کنید.

In [ ]:
import zipfile
archive=WORK/'SER_new_experiment_outputs.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in [EXPERIMENT, FEATURES]:
        if folder.exists():
            for p in folder.rglob('*'):
                if p.is_file() and p.name not in {'features.npz','features_incomplete.npz'}:
                    z.write(p,p.relative_to(WORK))
print(archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
